- [ ] Trace through `coprime_reconstruct_frequency` for inputs $M=7$, $N=11$, $k_n=5$, and $k_m=3$ and verify your
    hand calculations from parts a) and b). Make sure to color-code output using the color scheme specified below:
   - $(M,N)$ - (`COLOR_PALETTE["electric_blue"]`)
   - $(k_m, k_n)$ - `COLOR_PALETTE["coral_orange"]`
   - $k_\text{true}$ - `COLOR_PALETTE["vivid_pink"]'`
   - $(a,b)$ - `COLOR_PALETTE["brigh_green"]`

In [1]:
from __future__ import annotations

from kuramoto.cli.style import (
    PRESETS,
    cli_divider,
    cli_echo,
    cli_section,
    multi_gradient,
    style_coprime_coeff,
    style_coprime_gcd,
    style_coprime_remainder,
    style_coprime_term,
    style_crt_channel,
)
from kuramoto.solvers.coprime import coprime_reconstruct_frequency, extended_gcd


def trace_crt_reconstruction(
    k_m: int,
    k_n: int,
    M: int,
    N: int,
) -> int:
    """
    Trace the Chinese Remainder Theorem reconstruction step by step.

    Given coprime strides M and N and observed frequency aliases k_m (mod M)
    and k_n (mod N), this function prints a color-coded trace of the CRT
    formula and returns the reconstructed true frequency index.

    Parameters
    ----------
    k_m : int
        Observed alias modulo M (from Sampler B).
    k_n : int
        Observed alias modulo N (from Sampler A).
    M : int
        Subsampling stride for Sampler B (coprime to N).
    N : int
        Subsampling stride for Sampler A (coprime to M).

    Returns
    -------
    int
        The reconstructed true frequency index k_true in [0, M*N - 1].
    """
    modulus = M * N
    cli_section(f"CRT Reconstruction: ({style_coprime_term(M)}, {style_coprime_term(N)})")

    # Step 1: Bézout identity
    label_bezout = multi_gradient("Step 1 — Bézout Identity:", PRESETS["eigenscribe-2"])
    cli_echo(f"  {label_bezout}")
    g, a, b = extended_gcd(M, N)
    cli_echo(
        f"    ({style_coprime_coeff(a)}){style_coprime_term(M)} + ({style_coprime_coeff(b)}){style_coprime_term(N)} = {style_coprime_gcd(g)}"
    )
    cli_echo(
        f"    Bézout coefficients: {style_coprime_coeff('a')}={style_coprime_coeff(a)}, "
        f"{style_coprime_coeff('b')}={style_coprime_coeff(b)}"
    )
    cli_divider()

    # Step 2: Channel A contribution (k_n * a * M)
    label_a = multi_gradient("Step 2 — Channel A contribution:", PRESETS["eigenscribe-2"])
    cli_echo(f"  {label_a}")
    term_a = k_n * a * M
    cli_echo(
        f"    Term A = {style_coprime_remainder(k_n)} × ({style_coprime_coeff(a)}) × {style_coprime_term(M)} "
        f"= {style_crt_channel(term_a, 'A')}"
    )
    cli_echo(
        f"    (Recovers remainder mod {style_coprime_term(N)}: "
        f"{style_coprime_coeff(a)} × {style_coprime_term(M)} ≡ 1 (mod {style_coprime_term(N)}))"
    )
    cli_divider()

    # Step 3: Channel B contribution (k_m * b * N)
    label_b = multi_gradient("Step 3 — Channel B contribution:", PRESETS["eigenscribe-2"])
    cli_echo(f"  {label_b}")
    term_b = k_m * b * N
    cli_echo(
        f"    Term B = {style_coprime_remainder(k_m)} × ({style_coprime_coeff(b)}) × {style_coprime_term(N)} "
        f"= {style_crt_channel(term_b, 'B')}"
    )
    cli_echo(
        f"    (Recovers remainder mod {style_coprime_term(M)}: "
        f"{style_coprime_coeff(b)} × {style_coprime_term(N)} ≡ 1 (mod {style_coprime_term(M)}))"
    )
    cli_divider()

    # Step 4: Merge
    label_merge = multi_gradient("Step 4 — Merge:", PRESETS["eigenscribe-2"])
    cli_echo(f"  {label_merge}")
    k_true = (term_a + term_b) % modulus
    cli_echo(
        f"    k_true = ({style_crt_channel(term_a, 'A')} + {style_crt_channel(term_b, 'B')}) % {style_coprime_term(modulus)} "
        f"= {style_coprime_gcd(k_true)}"
    )
    cli_divider()

    # Step 5: Verification
    label_verify = multi_gradient("Step 5 — Verification:", PRESETS["eigenscribe-2"])
    cli_echo(f"  {label_verify}")
    check_m = k_true % M
    check_n = k_true % N
    match_m = "✔️" if check_m == k_m else "✖️"
    match_n = "✔️" if check_n == k_n else "✖️"
    cli_echo(
        f"    {style_coprime_gcd(k_true)} mod {style_coprime_term(M)} = {style_coprime_gcd(check_m)}  "
        f"{match_m} (matches k_m={style_coprime_remainder(k_m)})"
    )
    cli_echo(
        f"    {style_coprime_gcd(k_true)} mod {style_coprime_term(N)} = {style_coprime_gcd(check_n)}  "
        f"{match_n} (matches k_n={style_coprime_remainder(k_n)})"
    )
    cli_divider()

    # Cross-check with the library function
    k_lib = coprime_reconstruct_frequency(k_m=k_m, k_n=k_n, M=M, N=N)
    lib_match = "✔️" if k_lib == k_true else "✖️"
    cli_echo(
        f"  Library coprime_reconstruct_frequency() = {style_coprime_gcd(k_lib)}  {lib_match}"
    )

    return k_true


def trace_crt_reconstruction_error(
    k_m: int,
    k_n: int,
    M: int,
    N: int,
) -> None:
    """
    Trace what happens when M and N are not coprime.

    Demonstrates why coprimality is required for unique CRT reconstruction.
    """
    cli_section(f"CRT Reconstruction (Non-Coprime): ({style_coprime_term(M)}, {style_coprime_term(N)})")
    g, a, b = extended_gcd(M, N)
    cli_echo(
        f"  gcd({style_coprime_term(M)}, {style_coprime_term(N)}) = {style_coprime_gcd(g)}"
    )
    if g != 1:
        cli_echo(
            f"  ✖️ Strides are not coprime (gcd = {style_coprime_gcd(g)}). "
            f"CRT reconstruction is ambiguous."
        )
        cli_echo(
            f"  Multiple indices in [0, {M * N - 1}] share the same remainder pair "
            f"({style_coprime_remainder(k_m)}, {style_coprime_remainder(k_n)})."
        )
        # Show the ambiguous indices
        ambiguous = [t for t in range(M * N) if t % M == k_m and t % N == k_n]
        if ambiguous:
            cli_echo(f"  Matching indices: {ambiguous}")
        else:
            cli_echo(f"  No index matches — the remainder pair is inconsistent.")
    else:
        cli_echo(f"  ✔️ Strides are coprime. Proceeding with reconstruction.")
    cli_divider()

    # Attempt reconstruction — should raise ValueError
    try:
        k_true = coprime_reconstruct_frequency(k_m=k_m, k_n=k_n, M=M, N=N)
        cli_echo(f"  Result: {style_coprime_gcd(k_true)}")
    except ValueError as exc:
        cli_echo(f"  ValueError: {exc}")


 ──────────────────────────────────────────────────
  CRT Reconstruction: (7, 11)
 ──────────────────────────────────────────────────
  Step 1 — Bézout Identity:
    (-3)7 + (2)11 = 1
    Bézout coefficients: a=-3, b=2
 ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─
  Step 2 — Channel A contribution:
    Term A = 5 × (-3) × 7 = -105
    (Recovers remainder mod 11: -3 × 7 ≡ 1 (mod 11))
 ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─
  Step 3 — Channel B contribution:
    Term B = 3 × (2) × 11 = 66
    (Recovers remainder mod 7: 2 × 11 ≡ 1 (mod 7))
 ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─
  Step 4 — Merge:
    k_true = (-105 + 66) % 77 = 38
 ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─
  Step 5 — Verification:
    38 mod 7 = 3  ✔️ (matches k_m=3)
    38 mod 11 = 5  ✔️ (matches k_n=5)
 ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─
  Library coprime_reconstruct_frequency() = 38  ✔️

Reconstructed k_true = 38

 ──────────────────────────────────────────────────
  C

In [2]:
# Problem 17 example: k_n=5 (mod 11), k_m=3 (mod 7)
k_true = trace_crt_reconstruction(k_m=3, k_n=5, M=7, N=11)
print(f"\nReconstructed k_true = {k_true}")


 ──────────────────────────────────────────────────
  CRT Reconstruction: (7, 11)
 ──────────────────────────────────────────────────
  Step 1 — Bézout Identity:
    (-3)7 + (2)11 = 1
    Bézout coefficients: a=-3, b=2
 ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─
  Step 2 — Channel A contribution:
    Term A = 5 × (-3) × 7 = -105
    (Recovers remainder mod 11: -3 × 7 ≡ 1 (mod 11))
 ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─
  Step 3 — Channel B contribution:
    Term B = 3 × (2) × 11 = 66
    (Recovers remainder mod 7: 2 × 11 ≡ 1 (mod 7))
 ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─
  Step 4 — Merge:
    k_true = (-105 + 66) % 77 = 38
 ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─
  Step 5 — Verification:
    38 mod 7 = 3  ✔️ (matches k_m=3)
    38 mod 11 = 5  ✔️ (matches k_n=5)
 ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─
  Library coprime_reconstruct_frequency() = 38  ✔️

Reconstructed k_true = 38


In [3]:
# Non-coprime counterexample
trace_crt_reconstruction_error(k_m=0, k_n=0, M=6, N=9)


 ──────────────────────────────────────────────────
  CRT Reconstruction (Non-Coprime): (6, 9)
 ──────────────────────────────────────────────────
  gcd(6, 9) = 3
  ✖️ Strides are not coprime (gcd = 3). CRT reconstruction is ambiguous.
  Multiple indices in [0, 53] share the same remainder pair (0, 0).
  Matching indices: [0, 18, 36]
 ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─
  ValueError: Strides M=6 and N=9 are not coprime (gcd = 3). CRT frequency reconstruction requires gcd(M, N) == 1.


In [4]:
# Another coprime example
k_true_2 = trace_crt_reconstruction(k_m=2, k_n=7, M=5, N=8)
print(f"\nReconstructed k_true = {k_true_2}")


 ──────────────────────────────────────────────────
  CRT Reconstruction: (5, 8)
 ──────────────────────────────────────────────────
  Step 1 — Bézout Identity:
    (-3)5 + (2)8 = 1
    Bézout coefficients: a=-3, b=2
 ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─
  Step 2 — Channel A contribution:
    Term A = 7 × (-3) × 5 = -105
    (Recovers remainder mod 8: -3 × 5 ≡ 1 (mod 8))
 ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─
  Step 3 — Channel B contribution:
    Term B = 2 × (2) × 8 = 32
    (Recovers remainder mod 5: 2 × 8 ≡ 1 (mod 5))
 ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─
  Step 4 — Merge:
    k_true = (-105 + 32) % 40 = 7
 ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─
  Step 5 — Verification:
    7 mod 5 = 2  ✔️ (matches k_m=2)
    7 mod 8 = 7  ✔️ (matches k_n=7)
 ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─
  Library coprime_reconstruct_frequency() = 7  ✔️

Reconstructed k_true = 7
